In [0]:
from datetime import datetime

In [0]:
dbutils.widgets.text("batch_id" , "1" , "Batch Id 1 ,2 or 3")

In [0]:
from datetime import datetime

batch_id = dbutils.widgets.get("batch_id")
team_name  = "team_lemma"
bronze_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"

try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {staging_db}.finwire_parsed LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "Batch1"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "Batch1"
    
run_id=carried_run_id
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
if batch_id != "1":
    print("finwire has batch 1 only")
    dbutils.notebook.exit("finwire only has batch 1")

In [0]:
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit,row_number ,when
from pyspark.sql.types import DecimalType, LongType
from pyspark.sql.window import Window

In [0]:
df_staging = spark.table(f"{staging_db}.finwire_parsed")

    # Split into 3 logical groups
df_cmp = df_staging.filter(col("RecType") == "CMP")

df_sec = df_staging.filter(
        col("RecType").isin("SEC_CIK", "SEC_NAME")
    )

df_fin = df_staging.filter(
        col("RecType").isin("FIN_COMPANYID", "FIN_NAME")
    )

print(f"CMP rows : {df_cmp.count()}")
print(f"SEC rows : {df_sec.count()}")
print(f"FIN rows : {df_fin.count()}")

In [0]:
w_cmp = Window.partitionBy("CIK" ).orderBy(col("PTS").desc())

silver_company = (
        df_cmp
        .withColumn("_rn", row_number().over(w_cmp))
        .filter(col("_rn") == 1)
        .drop("_rn")
        .select(
            col("CIK").cast(LongType()).alias("companyid"),
            col("PTS").cast("date").alias("effectivedate"),
            trim(col("CompanyName")).alias("companyname"),
            trim(col("Status")).alias("status"),
            trim(col("IndustryID")).alias("industryid"),
            trim(col("SPrating")).alias("sprating"),
            col("FoundingDate").alias("foundingdate"),
            trim(col("AddrLine1")).alias("addressline1"),
            trim(col("AddrLine2")).alias("addressline2"),
            trim(col("PostalCode")).alias("postalcode"),
            trim(col("City")).alias("city"),
            trim(col("StateProvince")).alias("stateprov"),
            trim(col("Country")).alias("country"),
            trim(col("CEOname")).alias("ceo"),
            trim(col("Description")).alias("description"),
            lit(batch_id).alias("_batch"),
            lit(run_id).alias("_run_id"),
            current_timestamp().alias("_load_ts"),
        )
    )

print(f"Silver company rows to merge : {silver_company.count()}")
silver_company.printSchema()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Row

In [0]:
## write in sliver

recon_results = []

silver_company.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_db}.company")


count = spark.table(f"{silver_db}.company").count()
print(f"company count {count}")

In [0]:
recon_results = []





company_count = spark.table(f"{silver_db}.company").count()
print(f"company count  {company_count}")

recon_results.append(Row(
        source_table = "company",
        batch_id     = f"Batch{batch_id}",
        source_count = silver_company.count(),
        target_count = company_count,
        status       = "PASS"
    ))




In [0]:
### build security table 

company_lookup = (
    spark.table(f"{silver_db}.company")
    .select(
        col("companyid").alias("lkp_companyid"),
        col("companyname").alias("lkp_name")
        
    )
)

sec_resolved = (
    df_sec
    .join(company_lookup,
          trim(col("CoNameOrCIK")) == trim(col("lkp_name")),
          how="left")
    .withColumn("resolved_companyid",
        when(col("RecType") == "SEC_CIK",
             trim(col("CoNameOrCIK")).cast(LongType()))
        .otherwise(col("lkp_companyid"))  
    )
    .drop("lkp_companyid", "lkp_name")
)




w_sec = Window.partitionBy("Symbol").orderBy(col("PTS").desc())

silver_security = (
        sec_resolved
        .withColumn("_rn", row_number().over(w_sec))
        .filter(col("_rn") == 1)
        .drop("_rn")
        .select(
            trim(col("CoNameOrCIK")).alias("conameorcik"),
            trim(col("symbol")).alias("symbol"),
            trim(col("IssueType")).alias("issue"),
            trim(col("Status")).alias("status"),
            trim(col("SecurityName")).alias("name"),
            trim(col("ExID")).alias("exchangeid"),
            trim(col("SharesOutstanding")).cast("bigint").alias("sharesoutstanding"),
            trim(col("Dividend")).cast("decimal(10 , 2 )").alias("dividend"),
            trim(col("FirstTradeDate")).alias("firsttradedate"),
            trim(col("FirstTradeExchange")).alias("firsttradeexchange"),
            col("RecType").alias("rectype"),
            col("PTS").cast("date").alias("effectivedate"),
            col("resolved_companyid").cast(LongType()).alias("companyid"),
            lit(batch_id).alias("_batch"),
            lit(run_id).alias("_run_id"),
            current_timestamp().alias("_load_ts")
        )
    )

print(f"Silver security rows : {silver_security.count()}")
silver_security.printSchema()



In [0]:
## write in sliver



silver_security.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_db}.security") \
    


count = spark.table(f"{silver_db}.security").count()
print(f"company count {count}")

In [0]:
recon_results = []





security_count = spark.table(f"{silver_db}.security").count()
print(f"security count   {security_count}")

recon_results.append(Row(
        source_table = "security",
        batch_id     = f"Batch{batch_id}",
        source_count = silver_security.count(),
        target_count = security_count,
        status       = "PASS"
    ))




In [0]:

fin_resolved = (
    df_fin
    .join(company_lookup,
          trim(col("CoNameOrCIK")) == trim(col("lkp_name")),
          how="left")
    .withColumn("resolved_companyid",
        when(col("RecType") == "FIN_COMPANYID",
             trim(col("CoNameOrCIK")).cast(LongType()))
        .otherwise(col("lkp_companyid"))
    )
    .drop("lkp_companyid", "lkp_name")
)

silver_financial = (
    fin_resolved.select(
        col("resolved_companyid").cast(LongType()).alias("companyid"),
        trim(col("CoNameOrCIK")).alias("conameorcik"),
        col("RecType").alias("rectype"),
        col("FI_Year").cast("int").alias("year"),
        col("FI_Quarter").cast("int").alias("quarter"),
        col("QtrStartDate").alias("fi_qtr_start_date"),
        col("Revenue").cast("decimal(15,2)").alias("fi_revenue"),
        col("Earnings").cast("decimal(15,2)").alias("fi_net_earn"),
        col("EPS").cast("decimal(10,2)").alias("fi_basic_eps"),
        col("DilutedEPS").cast("decimal(10,2)").alias("fi_dilut_eps"),
        col("Margin").cast("decimal(10,2)").alias("fi_margin"),
        col("Inventory").cast("decimal(15,2)").alias("fi_inventory"),
        col("Assets").cast("decimal(15,2)").alias("fi_assets"),
         col("Liabilities").cast("decimal(15,2)").alias("fi_liability"),
        col("SharesOutstanding").cast("bigint").alias("fi_out_basic"),
        col("DilutedSharesOut").cast("bigint").alias("fi_out_dilut"),
        col("PTS").cast("date").alias("effectivedate"),
        lit(batch_id).alias("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts")
        
    )
)

print("financial count : ",silver_financial.count())
silver_financial.printSchema()


In [0]:
silver_financial.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_db}.financial")\

count = spark.table(f"{silver_db}.financial").count()
print(f"financial count {count}")

In [0]:
recon_results = []

financial_count = spark.table(f"{silver_db}.financial").count()
print(f"financial count   {financial_count}")

recon_results.append(Row(
        source_table = "financial",
        batch_id     = f"Batch{batch_id}",
        source_count = silver_financial.count(),
        target_count = financial_count,
        status       = "PASS"
    ))




In [0]:
checks = {
    "silver.company"  : spark.table(f"{silver_db}.company").count(),
    "silver.security" : spark.table(f"{silver_db}.security").count(),
    "silver.financial": spark.table(f"{silver_db}.financial").count()
}

EXPECTED = {
    "silver.company"  : 4596,
    "silver.security" : 7598,
    "silver.financial": 457025
}

print(f"\n{'Table':<25} {'Actual':>10} {'Expected':>10} {'Status'}")
print("-" * 60)
for table, actual in checks.items():
    expected = EXPECTED[table]
    status   = "PASS" if actual == expected else "FAIL"
    print(f"{table:<25} {actual:>10} {expected:>10} {status}")

In [0]:
%run ../../02_common_utils/operations

In [0]:
recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if "ERROR" not in row.status:
        log_pipeline_recon(
            spark        = spark,
            run_id       = run_id,
            batch_id     = row.batch_id,
            domain       = "MARKET",
            table_name   = row.source_table,
            source_layer = "staging",
            target_layer = "silver",
            source_count = row.source_count,
            target_count = row.target_count
        )

        log_audit_event(
            spark         = spark,
            run_id        = run_id,
            batch         = row.batch_id,
            layer         = "silver",
            table_name    = row.source_table,
            operation     = "MERGE",
            rows_affected = row.target_count
        )

display(recon_df)